In [ ]:
import requests

patient_id=30358
server_url = "http://hapi.fhir.org/baseR4/DiagnosticReport?subject=Patient/" + str(patient_id)

try:
    response = requests.get(server_url)
    if response.status_code == 200:
        historial = response.json()
        print(historial['total'])

        if "entry" in historial:
            # Itera a través de las entradas (DiagnosticReport)
            for entry in historial["entry"]:
                resource = entry.get("resource")
                if resource and resource.get("resourceType") == "DiagnosticReport":
                    # Obtiene los datos de interés
                    last_updated = resource["meta"]["lastUpdated"]
                    code = resource["code"]["coding"][0]["code"]
                    status = resource["status"]
                    source = resource["meta"]["source"]
                    
                    # Imprime los datos
                    print(f"lastUpdated: {last_updated}")
                    print(f"code: {code}")
                    print(f"status: {status}")
                    print(f"source: {source}")
                    print("\n")
        
except:
    pass


In [35]:
###Obtener las modalidades


import requests

server_url = "http://127.0.0.1:8080/fhir/ValueSet"
response = requests.get(server_url)
if response.status_code == 200:
    respuesta = response.json()
    print(respuesta['entry'])
    modalities = []
    if 'entry' in respuesta:
        for entry in respuesta['entry']:
            print(entry['resource']['compose']['include'][0]['concept'])
            for concept in entry['resource']['compose']['include'][0]['concept']:
                if 'code' in concept:
                        modalities.append(concept['code'])

    print(modalities)    

[{'fullUrl': 'http://127.0.0.1:8080/fhir/ValueSet/1591', 'resource': {'resourceType': 'ValueSet', 'id': '1591', 'meta': {'versionId': '1', 'lastUpdated': '2023-10-30T03:32:36.519+00:00', 'source': '#15TmCc9Wu4cwQivA'}, 'url': 'http://example.org/valuesets/specialties', 'status': 'active', 'compose': {'include': [{'system': 'http://example.org/specialties', 'concept': [{'code': 'mamo', 'display': 'Mamografía'}, {'code': 'radio', 'display': 'Radiología'}, {'code': 'eco', 'display': 'Ecografía'}, {'code': 'Reso', 'display': 'Resonancia'}, {'code': 'tomo', 'display': 'Tomografía'}]}]}}, 'search': {'mode': 'match'}}]
[{'code': 'mamo', 'display': 'Mamografía'}, {'code': 'radio', 'display': 'Radiología'}, {'code': 'eco', 'display': 'Ecografía'}, {'code': 'Reso', 'display': 'Resonancia'}, {'code': 'tomo', 'display': 'Tomografía'}]
['mamo', 'radio', 'eco', 'Reso', 'tomo']


In [34]:
###Obtener las partes del cuerpo


import requests

server_url = "http://127.0.0.1:8080/fhir/CodeSystem"
response = requests.get(server_url)
if response.status_code == 200:
    respuesta = response.json()
    print(respuesta['entry'])
    partesdelbody = []

    for entry in respuesta['entry']:
        
        for concept in entry['resource']['concept']:
            if 'code' in concept:
                    partesdelbody.append(concept['code'])

    print(partesdelbody)    

[{'fullUrl': 'http://127.0.0.1:8080/fhir/CodeSystem/1588', 'resource': {'resourceType': 'CodeSystem', 'id': '1588', 'meta': {'versionId': '1', 'lastUpdated': '2023-10-30T01:58:31.940+00:00', 'source': '#KCSJvYrag4qtXtmX'}, 'url': 'http://example.org/codesystems/body-part', 'version': '1.0', 'name': 'Partes del cuerpo', 'title': 'Partes del cuerpo', 'status': 'active', 'content': 'complete', 'concept': [{'code': 'Abdomen', 'display': 'Abdomen'}, {'code': 'Abdomen y Pelvis', 'display': 'Abdomen y Pelvis'}, {'code': 'Cabeza y cuello', 'display': 'Cabeza y cuello'}, {'code': 'Columna Vertebral', 'display': 'Columna Vertebral'}, {'code': 'Cuerpo Entero', 'display': 'Cuerpo Entero'}, {'code': 'Extremidades inferiores', 'display': 'Extremidades inferiores'}, {'code': 'Extremidades superiores', 'display': 'Extremidades superiores'}, {'code': 'Pelvis', 'display': 'Pelvis'}, {'code': 'Torax', 'display': 'Torax'}, {'code': 'Torax y abdomen', 'display': 'Torax y Abdomen'}]}, 'search': {'mode': 'ma

In [37]:
def crear_recurso_diagnostic_report(parte_cuerpo, estudio):
    return {
        "resourceType": "DiagnosticReportDefinition",
        "status": "active",
        "code": {
            "coding": [
                {
                    "system": "http://example.org/specialties",
                    "code": f"{estudio}-{parte_cuerpo.lower()}",
                    "display": f"{estudio.capitalize()} de {parte_cuerpo}"
                }
            ],
            "text": f"{estudio.capitalize()} de {parte_cuerpo}"
        },
        "subjectCodeableConcept": {
            "coding": [
                {
                    "system": "http://example.org/codesystems/body-part",
                    "code": parte_cuerpo,
                    "display": parte_cuerpo
                }
            ],
            "text": parte_cuerpo
        },
        "category": [
            {
                "coding": [
                    {
                        "system": "http://example.org/modalities",
                        "code": estudio,
                        "display": estudio.capitalize()
                    }
                ],
                "text": estudio.capitalize()
            }
        ]
    }

In [38]:
for modalidad in modalities:
    for parte in partesdelbody:
        estudio=crear_recurso_diagnostic_report(parte, modalidad)
        print(estudio)

{'resourceType': 'DiagnosticReportDefinition', 'status': 'active', 'code': {'coding': [{'system': 'http://example.org/specialties', 'code': 'mamo-abdomen', 'display': 'Mamo de Abdomen'}], 'text': 'Mamo de Abdomen'}, 'subjectCodeableConcept': {'coding': [{'system': 'http://example.org/codesystems/body-part', 'code': 'Abdomen', 'display': 'Abdomen'}], 'text': 'Abdomen'}, 'category': [{'coding': [{'system': 'http://example.org/modalities', 'code': 'mamo', 'display': 'Mamo'}], 'text': 'Mamo'}]}
{'resourceType': 'DiagnosticReportDefinition', 'status': 'active', 'code': {'coding': [{'system': 'http://example.org/specialties', 'code': 'mamo-abdomen y pelvis', 'display': 'Mamo de Abdomen y Pelvis'}], 'text': 'Mamo de Abdomen y Pelvis'}, 'subjectCodeableConcept': {'coding': [{'system': 'http://example.org/codesystems/body-part', 'code': 'Abdomen y Pelvis', 'display': 'Abdomen y Pelvis'}], 'text': 'Abdomen y Pelvis'}, 'category': [{'coding': [{'system': 'http://example.org/modalities', 'code': '

In [41]:
import requests

server_url = "http://127.0.0.1:8080/fhir/DiagnosticReportDefinition"
for modalidad in modalities:
    for parte in partesdelbody:
        estudio=crear_recurso_diagnostic_report(parte, modalidad)
        
        response=requests.post(server_url,estudio)
        print(response.status_code)
        if response.status_code==200:
            print("piola")


404
404
404
404
404
404
404
404
404
404
404
404
404
404
404
404
404
404
404
404
404
404
404
404
404
404
404
404
404
404
404
404
404
404
404
404
404
404
404
404
404
404
404
404
404
404
404
404
404
404


In [88]:
import sqlite3

query="CREATE TABLE TipoEstudio (\
    CodigoTipoEstudio VARCHAR(10) PRIMARY KEY,\
    Descripcion VARCHAR(255),\
    Modo VARCHAR(50),\
    ParteCuerpo VARCHAR(255),\
    FOREIGN KEY (Modo) REFERENCES Modalidad(NombreModalidad),\
    FOREIGN KEY (ParteCuerpo) REFERENCES PartesCuerpo(NombreParteCuerpo));"

conn = sqlite3.connect('basededatos.db')
cursor = conn.cursor()
cursor.execute(query)
conn.commit()
conn.close()


In [120]:
import sqlite3

query="DROP TABLE Machine"

conn = sqlite3.connect('basededatos.db')
cursor = conn.cursor()
cursor.execute(query)
conn.commit()
conn.close()

In [44]:
query="CREATE TABLE PartesCuerpo (\
    NombreParteCuerpo VARCHAR(255) PRIMARY KEY\
);"

conn = sqlite3.connect('basededatos.db')
cursor = conn.cursor()
cursor.execute(query)
conn.commit()
conn.close()

In [45]:
query="CREATE TABLE Modalidad (\
    NombreModalidad VARCHAR(255) PRIMARY KEY,\
    Modo VARCHAR(50)\
);"

conn = sqlite3.connect('basededatos.db')
cursor = conn.cursor()
cursor.execute(query)
conn.commit()
conn.close()

In [123]:
query="SELECT * FROM Machine"
conn = sqlite3.connect('basededatos.db')
cursor = conn.cursor()
cursor.execute(query)
tablas=cursor.fetchall()
for dato in tablas:
    print(dato)

(1, 'Eco00', 'Ecografia', '1600', 'Eco00')
(2, 'Eco01', 'Ecografia', '1601', 'Eco01')
(3, 'Eco02', 'Ecografia', '1602', 'Eco02')
(4, 'Eco03', 'Ecografia', '1603', 'Eco03')
(5, 'Eco04', 'Ecografia', '1604', 'Eco04')
(6, 'MAM05', 'MAMOGRAFIA', '1605', 'MAM05')
(7, 'MAM06', 'MAMOGRAFIA', '1606', 'MAM06')
(8, 'MAM07', 'MAMOGRAFIA', '1607', 'MAM07')
(9, 'RAD08', 'RADIOLOGIA', '1608', 'RAD08')
(10, 'RAD09', 'RADIOLOGIA', '1609', 'RAD09')
(11, 'RAD10', 'RADIOLOGIA', '1610', 'RAD10')
(12, 'RAD11', 'RADIOLOGIA', '1611', 'RAD11')
(13, 'RAD12', 'RADIOLOGIA', '1612', 'RAD12')
(14, 'RAD13', 'RADIOLOGIA', '1613', 'RAD13')
(15, 'RAD14', 'RADIOLOGIA', '1614', 'RAD14')
(16, 'RES15', 'RESONANCIA', '1615', 'RES15')
(17, 'RES16', 'RESONANCIA', '1616', 'RES16')
(18, 'RES17', 'RESONANCIA', '1617', 'RES17')
(19, 'RES18', 'RESONANCIA', '1618', 'RES18')
(20, 'TOM19', 'TOMOGRAFIA', '1619', 'TOM19')
(21, 'TOM20', 'TOMOGRAFIA', '1620', 'TOM20')
(22, 'TOM21', 'TOMOGRAFIA', '1621', 'TOM21')
(23, 'TOM22', 'TOMOGRAFI

In [68]:
partesdelbody

['Abdomen',
 'Abdomen y Pelvis',
 'Cabeza y cuello',
 'Columna Vertebral',
 'Cuerpo Entero',
 'Extremidades inferiores',
 'Extremidades superiores',
 'Pelvis',
 'Torax',
 'Torax y abdomen']

In [73]:
query="INSERT INTO Modalidad(NombreModalidad) VALUES (?)"
conn = sqlite3.connect('basededatos.db')
cursor = conn.cursor()
for modalidad in modalities:
    cursor.execute(query,(modalidad,))
    conn.commit()

In [82]:
query="INSERT INTO TipoEstudio(CodigoTipoEstudio,Descripcion,Modo,ParteCuerpo) VALUES (?,?,?,?)"
conn = sqlite3.connect('basededatos.db')
cursor = conn.cursor()
for modalidad in modalities:
    i=1
    for parte in partesdelbody:
        codigo=modalidad[0]+str(i)
        description=modalidad+" "+parte
        print(description)
        i+=1
        cursor.execute(query,(codigo,description,modalidad,parte,))
        conn.commit()



mamo Abdomen
mamo Abdomen y Pelvis
mamo Cabeza y cuello
mamo Columna Vertebral
mamo Cuerpo Entero
mamo Extremidades inferiores
mamo Extremidades superiores
mamo Pelvis
mamo Torax
mamo Torax y abdomen
radio Abdomen
radio Abdomen y Pelvis
radio Cabeza y cuello
radio Columna Vertebral
radio Cuerpo Entero
radio Extremidades inferiores
radio Extremidades superiores
radio Pelvis
radio Torax
radio Torax y abdomen
eco Abdomen
eco Abdomen y Pelvis
eco Cabeza y cuello
eco Columna Vertebral
eco Cuerpo Entero
eco Extremidades inferiores
eco Extremidades superiores
eco Pelvis
eco Torax
eco Torax y abdomen
Reso Abdomen
Reso Abdomen y Pelvis
Reso Cabeza y cuello
Reso Columna Vertebral
Reso Cuerpo Entero
Reso Extremidades inferiores
Reso Extremidades superiores
Reso Pelvis
Reso Torax
Reso Torax y abdomen
tomo Abdomen
tomo Abdomen y Pelvis
tomo Cabeza y cuello
tomo Columna Vertebral
tomo Cuerpo Entero
tomo Extremidades inferiores
tomo Extremidades superiores
tomo Pelvis
tomo Torax
tomo Torax y abdomen


In [107]:
import sqlite3
import csv
conn = sqlite3.connect('basededatos.db')
cursor = conn.cursor()
# Abre el archivo CSV y guarda los datos en la tabla
csv_file = 'D:/PROYECTOS/Hapi_Fhir_server_app_seed/tipoestudios.csv'
with open(csv_file, 'r') as csv_file:
    csv_reader = csv.reader(csv_file, delimiter=';')
    next(csv_reader)  # Salta la fila de encabezado si la tiene
    for row in csv_reader:
        print(row[0])
        # Ajusta esto para que coincida con la estructura de tu tabla
        cursor.execute('INSERT INTO TipoEstudio(CodigoTipoEstudio,Descripcion,Modo,ParteCuerpo) VALUES (?,?,?,?)', (row[0],row[1],row[2],row[3],))
        conn.commit()
        

# Guarda los cambios y cierra la conexión

conn.close()

ECO01
ECO02
ECO03
ECO04
ECO05
ECO06
ECO07
ECO08
ECO09
ECO10
ECO11
ECO12
ECO13
ECO14
ECO15
ECO16
ECO17
ECO18
ECO19
ECO20
ECO21
ECO22
ECO23
ECO24
ECO25
ECO26
ECO27
ECO28
ECO29
ECO30
ECO31
ECO32
ECO33
ECO34
ECO35
ECO36
ECO37
ECO38
ECO39
ECO40
ECO41
ECO42
ECO43
ECO44
ECO45
ECO46
ECO47
ECO48
ECO49
ECO50
ECO51
ECO52
ECO53
ECO54
ECO55
ECO56
ECO57
ECO58
ECO59
ECO60
ECO61
ECO62
ECO63
ECO64
ECO65
ECO66
ECO67
ECO68
ECO69
ECO70
ECO71
ECO72
ECO73
ECO74
ECO75
ECO76
ECO77
ECO78
ECO79
ECO80
ECO81
ECO82
ECO83
ECO84
MAM01
MAM02
MAM03
MAM04
MAM05
MAM06
MAM07
MAM08
MAM09
MAM10
MAM11
RAD01
RAD02
RAD03
RAD04
RAD05
RAD06
RAD07
RAD08
RAD09
RAD10
RAD11
RAD12
RAD13
RAD14
RAD15
RAD16
RAD17
RAD18
RAD19
RAD20
RAD21
RAD22
RAD23
RAD24
RAD25
RAD26
RAD27
RAD28
RAD29
RAD30
RAD31
RAD32
RAD33
RAD34
RAD35
RAD36
RAD37
RAD38
RAD39
RAD40
RAD41
RAD42
RAD43
RAD44
RAD45
RAD46
RAD47
RAD48
RAD49
RAD50
RAD51
RAD52
RAD53
RAD54
RAD55
RAD56
RAD57
RAD58
RAD59
RAD60
RAD61
RAD62
RAD63
RAD64
RAD65
RAD66
RAD67
RAD68
RAD69
RAD70
RAD71
RAD7

In [121]:
query="CREATE TABLE Machine (\
    idModalidad INTEGER PRIMARY KEY AUTOINCREMENT,\
    codigo VARCHAR(50),\
    Tipo VARCHAR(50),\
    idFHIR VARCHAR(50),\
    idAgenda VARCHAR(50));"
#Codigo será el asignado por la empresa o clinica
#idFHIR 
conn = sqlite3.connect('basededatos.db')
cursor = conn.cursor()
cursor.execute(query)
conn.commit()
conn.close() 

In [114]:
import csv

modalidades = ['Ecografia', 'MAMOGRAFIA', 'RADIOLOGIA', 'RESONANCIA', 'TOMOGRAFIA']
num_equipos_por_modalidad = [5, 3, 7, 4, 6]

# Abre un archivo CSV en modo escritura
with open('datos_machine.csv', 'w', newline='') as csvfile:
    fieldnames = ['codigo', 'Tipo', 'idFHIR', 'idAgenda']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    
    # Escribe la fila de encabezado
    writer.writeheader()

    equipo_counter = 0  # Para llevar la cuenta de equipos por modalidad
    for i, modalidad in enumerate(modalidades):
        for _ in range(num_equipos_por_modalidad[i]):
            codigo = f'{modalidad[:3]}{equipo_counter:02}'
            idFHIR = str(1600 + equipo_counter)
            idAgenda = codigo
            writer.writerow({'codigo': codigo, 'Tipo': modalidad, 'idFHIR': idFHIR, 'idAgenda': idAgenda})
            equipo_counter += 1

print("Archivo 'datos_machine.csv' generado exitosamente.")


Archivo 'datos_machine.csv' generado exitosamente.


In [122]:
import sqlite3
import csv
conn = sqlite3.connect('basededatos.db')
cursor = conn.cursor()
# Abre el archivo CSV y guarda los datos en la tabla
csv_file = 'D:/PROYECTOS/Hapi_Fhir_server_app_seed/flask-argon-dashboard-master/datos_machine.csv'
with open(csv_file, 'r') as csv_file:
    csv_reader = csv.reader(csv_file, delimiter=',')
    next(csv_reader)  # Salta la fila de encabezado si la tiene
    for row in csv_reader:
        print(row)
        # Ajusta esto para que coincida con la estructura de tu tabla
        cursor.execute('INSERT INTO Machine(codigo,Tipo,idFHIR,idAgenda) VALUES (?,?,?,?)', (row[0],row[1],row[2],row[3],))
        conn.commit()
        

# Guarda los cambios y cierra la conexión

conn.close()

['Eco00', 'Ecografia', '1600', 'Eco00']
['Eco01', 'Ecografia', '1601', 'Eco01']
['Eco02', 'Ecografia', '1602', 'Eco02']
['Eco03', 'Ecografia', '1603', 'Eco03']
['Eco04', 'Ecografia', '1604', 'Eco04']
['MAM05', 'MAMOGRAFIA', '1605', 'MAM05']
['MAM06', 'MAMOGRAFIA', '1606', 'MAM06']
['MAM07', 'MAMOGRAFIA', '1607', 'MAM07']
['RAD08', 'RADIOLOGIA', '1608', 'RAD08']
['RAD09', 'RADIOLOGIA', '1609', 'RAD09']
['RAD10', 'RADIOLOGIA', '1610', 'RAD10']
['RAD11', 'RADIOLOGIA', '1611', 'RAD11']
['RAD12', 'RADIOLOGIA', '1612', 'RAD12']
['RAD13', 'RADIOLOGIA', '1613', 'RAD13']
['RAD14', 'RADIOLOGIA', '1614', 'RAD14']
['RES15', 'RESONANCIA', '1615', 'RES15']
['RES16', 'RESONANCIA', '1616', 'RES16']
['RES17', 'RESONANCIA', '1617', 'RES17']
['RES18', 'RESONANCIA', '1618', 'RES18']
['TOM19', 'TOMOGRAFIA', '1619', 'TOM19']
['TOM20', 'TOMOGRAFIA', '1620', 'TOM20']
['TOM21', 'TOMOGRAFIA', '1621', 'TOM21']
['TOM22', 'TOMOGRAFIA', '1622', 'TOM22']
['TOM23', 'TOMOGRAFIA', '1623', 'TOM23']
['TOM24', 'TOMOGRAFIA

In [126]:
query="SELECT codigo FROM Machine"
conn = sqlite3.connect('basededatos.db')
cursor = conn.cursor()
cursor.execute(query)
tablas=cursor.fetchall()
codigos=[]
for dato in tablas:
    codigos.append(dato[0])

print(codigos)

['Eco00', 'Eco01', 'Eco02', 'Eco03', 'Eco04', 'MAM05', 'MAM06', 'MAM07', 'RAD08', 'RAD09', 'RAD10', 'RAD11', 'RAD12', 'RAD13', 'RAD14', 'RES15', 'RES16', 'RES17', 'RES18', 'TOM19', 'TOM20', 'TOM21', 'TOM22', 'TOM23', 'TOM24']


In [128]:

#Codigo será el asignado por la empresa o clinica
#idFHIR 
conn = sqlite3.connect('basededatos.db')
cursor = conn.cursor()

for codigo in codigos:
    query=f"CREATE TABLE AG_{codigo} (\
    idevento INTEGER PRIMARY KEY AUTOINCREMENT,\
    nombre VARCHAR(50),\
    fecha_inicio TEXT,\
    fecha_fin TEXT,\
    idAgenda VARCHAR(50));"
    #print(query)
    cursor.execute(query)
    conn.commit()
conn.close() 

In [132]:
machine = 'Eco01'
    
query=f"SELECT * FROM AG_{machine}"
conn = sqlite3.connect('basededatos.db')
cursor = conn.cursor()
cursor.execute(query)
tablas=cursor.fetchall()
for dato in tablas:
    print(dato)



(1, 'Eco abdomen con puncion', '2023-11-15 09:00:00', '2023-11-15 10:00:00', '123456789A')
(2, 'Eco Biliar', '2023-11-16 10:30:00', '2023-11-16 12:00:00', '987654321B')
(3, 'Eco Doppler Mesentérico', '2023-11-17 14:00:00', '2023-11-17 15:30:00', '456789123C')
(4, 'Ecografía Pancreática', '2023-11-18 08:45:00', '2023-11-18 10:15:00', '789123456D')
(5, 'ECO MUÑECA DERECHA', '2023-11-19 11:30:00', '2023-11-19 12:30:00', '321654987E')
(6, 'ECO DOPPLER MIEMBRO SUPERIOR DERECHO ARTERIAL', '2023-11-20 13:15:00', '2023-11-20 14:45:00', '654987321F')
(7, 'ECO MUÑECA IZQUIERDA', '2023-11-21 10:00:00', '2023-11-21 11:30:00', '234567891G')
(8, 'ECOGRAFIA ENDOANAL 360', '2023-11-22 15:00:00', '2023-11-22 16:30:00', '567891234H')
(9, 'ECO DOPPLER CARDIACA\t', '2023-11-23 09:30:00', '2023-11-23 10:45:00', '890123456I')
(10, 'Eco de Aorta abdominal', '2023-11-24 12:45:00', '2023-11-24 13:30:00', '678912345J')


In [130]:
eventos = [
    ("Eco abdomen con puncion", "2023-11-15 09:00:00", "2023-11-15 10:00:00", "123456789A"),
    ("Eco Biliar", "2023-11-16 10:30:00", "2023-11-16 12:00:00", "987654321B"),
    ("Eco Doppler Mesentérico", "2023-11-17 14:00:00", "2023-11-17 15:30:00", "456789123C"),
    ("Ecografía Pancreática", "2023-11-18 08:45:00", "2023-11-18 10:15:00", "789123456D"),
    ("ECO MUÑECA DERECHA", "2023-11-19 11:30:00", "2023-11-19 12:30:00", "321654987E"),
    ("ECO DOPPLER MIEMBRO SUPERIOR DERECHO ARTERIAL", "2023-11-20 13:15:00", "2023-11-20 14:45:00", "654987321F"),
    ("ECO MUÑECA IZQUIERDA", "2023-11-21 10:00:00", "2023-11-21 11:30:00", "234567891G"),
    ("ECOGRAFIA ENDOANAL 360", "2023-11-22 15:00:00", "2023-11-22 16:30:00", "567891234H"),
    ("ECO DOPPLER CARDIACA	", "2023-11-23 09:30:00", "2023-11-23 10:45:00", "890123456I"),
    ("Eco de Aorta abdominal", "2023-11-24 12:45:00", "2023-11-24 13:30:00", "678912345J")
]
conn = sqlite3.connect('basededatos.db')
cursor = conn.cursor()
for evento in eventos:
    cursor.execute("INSERT INTO AG_eco01 (nombre, fecha_inicio, fecha_fin, idAgenda) VALUES (?, ?, ?, ?)", evento)

# Asegúrate de realizar la confirmación de los cambios en la base de datos después de insertar los eventos
conn.commit()